# Compare dataset labels with trained SAM 3
Point `CHECKPOINT` at notebook 1's `best.pt`, enable inference, and run all cells.
This notebook needs only the dataset and saved checkpoint; it does not import or execute notebook 1.

Each row shows **original image | dataset labels | trained SAM 3**, using cyan for tobacco
and yellow for weeds. Instance predictions are combined by class for a direct comparison with
the dataset's semantic masks. Predictions use RGB and text prompts only, never the label mask.
The six default images come from held-out campaigns 7 and 8. These examples are qualitative;
training selected the checkpoint using campaign 6 validation IoU.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
from PIL import Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'notebooks/reproduction.py').is_file())
sys.path.insert(0, str(ROOT / 'notebooks'))
from reproduction import new_run


In [ ]:
RUN_INFERENCE = False  # Set True on the GPU machine, then run all cells.
DATA = ROOT / 'data/tobacco_original/Tobacco Aerial Dataset V2/Tobacco Aerial Dataset'
CHECKPOINT = ROOT / 'experiments/reproductions/sam3-simple-training-01/best.pt'
RUN_NAME = 'sam3-simple-examples-01'  # Fresh name for saved comparisons.
CONFIDENCE = 0.5
# None selects three evenly spaced patches from each held-out campaign (7 and 8).
# Or enter V2-relative RGB paths, e.g. 'Campaign no. 7/Patch images/data/1.png'.
EXAMPLE_IMAGES = None


## Load the trained model and predict
The saved full state dictionary is loaded strictly, without downloading weights. Use the training resolution stored in the checkpoint.

In [ ]:
def load_sam3(checkpoint, training=False):
    import torch
    from sam3.model_builder import build_sam3_image_model
    if not torch.cuda.is_available():
        raise RuntimeError('Run enabled SAM 3 cells on the CUDA GPU machine.')
    payload = torch.load(checkpoint, map_location='cpu', weights_only=True)
    state = payload.get('model', payload)
    # Original SAM 3 stores detector.*; our saved weights are already unprefixed.
    if any(k.startswith('detector.') for k in state):
        state = {k.removeprefix('detector.'): v for k, v in state.items()
                 if k.startswith('detector.')}
    # The original checkpoint also includes the optional interactive SAM 2 neck.
    # It is unused with enable_inst_interactivity=False; keep all other checks strict.
    state = {k: v for k, v in state.items()
             if not k.startswith('backbone.vision_backbone.sam2_convs.')}
    model = build_sam3_image_model(device='cpu', eval_mode=not training,
                                  load_from_HF=False, enable_segmentation=True,
                                  enable_inst_interactivity=False, compile=False)
    model.load_state_dict(state, strict=True)
    return model.cuda().train(training)


In [ ]:
def predict_classes(processor, image):
    """Predict from RGB and fixed text only; the highest-score mask owns overlaps."""
    state = processor.set_image(image)
    labels = np.zeros((image.height, image.width), dtype=np.uint8)
    confidence = np.zeros(labels.shape, dtype=np.float32)
    for category, prompt in ((1, 'tobacco plant'), (2, 'weed plant')):
        processor.reset_all_prompts(state)
        output = processor.set_text_prompt(prompt=prompt, state=state)
        for mask, score in zip(output['masks'], output['scores']):
            mask = mask.squeeze(0).detach().cpu().numpy().astype(bool)
            score = float(score)
            take = mask & (score > confidence)
            labels[take] = category
            confidence[take] = score
    return labels


## Select images and use matching colors
Change `EXAMPLE_IMAGES` to inspect particular images, or leave it as `None` for six deterministic examples.

In [ ]:
def image_pairs(data, campaigns):
    pairs = []
    for campaign in campaigns:
        folder = data / f'Campaign no. {campaign}' / ('Patch Images' if campaign == 2 else 'Patch images')
        images = sorted((folder / 'data').glob('*.png'), key=lambda p: int(p.stem))
        if not images:
            raise FileNotFoundError(f'No images in {folder / "data"}')
        for image in images:
            mask = folder / 'mask' / image.name
            if not mask.is_file():
                raise FileNotFoundError(mask)
            pairs.append((image, mask))
    return pairs


In [ ]:
def select_examples(data, requested=None):
    if requested is not None:
        paths = [data / Path(p) for p in requested]
        return [(p, p.parent.parent / 'mask' / p.name) for p in paths]
    pairs = []
    for campaign in (7, 8):
        candidates = image_pairs(data, [campaign])
        indices = np.linspace(0, len(candidates) - 1, 3, dtype=int)
        pairs.extend(candidates[i] for i in indices)
    return pairs


def overlay(image, labels):
    rgb = np.asarray(image, dtype=np.float32) / 255
    colors = {1: np.array([0.0, 0.85, 1.0]), 2: np.array([1.0, 0.85, 0.0])}
    for category, color in colors.items():
        selected = labels == category
        rgb[selected] = 0.55 * rgb[selected] + 0.45 * color
    return rgb


def comparison(image, truth, prediction, title):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, pixels, name in zip(axes, [image, overlay(image, truth), overlay(image, prediction)],
                                ['Original image', 'Dataset labels', 'Trained SAM 3']):
        ax.imshow(pixels)
        ax.set_title(name)
        ax.axis('off')
    fig.suptitle(title)
    fig.legend(handles=[Patch(color='#00d9ff', label='Tobacco'),
                        Patch(color='#ffd900', label='Weed')], loc='lower center', ncol=2)
    fig.tight_layout(rect=(0, 0.07, 1, 0.94))
    return fig


## Show and save comparisons
Adjust `CONFIDENCE` if needed. Save figures and class-ID masks under a fresh experiment name.

In [ ]:
if RUN_INFERENCE:
    import torch
    import matplotlib.pyplot as plt
    from sam3.model.sam3_image_processor import Sam3Processor
    if not 0 <= CONFIDENCE <= 1:
        raise ValueError('Confidence must be between 0 and 1.')
    payload = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    resolution = payload['options']['resolution']
    epoch = payload['epoch']
    del payload
    pairs = select_examples(DATA, EXAMPLE_IMAGES)
    if not pairs:
        raise ValueError('Select at least one example.')
    for image_path, mask_path in pairs:
        if not image_path.is_file() or not mask_path.is_file():
            raise FileNotFoundError(f'Missing example image or mask: {image_path}')
    model = load_sam3(CHECKPOINT)
    processor = Sam3Processor(model, resolution=resolution, confidence_threshold=CONFIDENCE)
    run = new_run(ROOT, RUN_NAME)
    (run / 'config.json').write_text(json.dumps({
        'checkpoint': str(CHECKPOINT), 'epoch': epoch, 'confidence': CONFIDENCE,
        'examples': [p.relative_to(DATA).as_posix() for p, _ in pairs]}, indent=2))
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
        for index, (image_path, mask_path) in enumerate(pairs, 1):
            image = Image.open(image_path).convert('RGB')
            truth = np.asarray(Image.open(mask_path))
            prediction = predict_classes(processor, image)
            title = image_path.relative_to(DATA).as_posix()
            fig = comparison(image, truth, prediction, title)
            fig.savefig(run / f'comparison_{index:02d}.png', dpi=150, bbox_inches='tight')
            Image.fromarray(prediction).save(run / f'prediction_{index:02d}.png')
            plt.show()
            plt.close(fig)
    print(f'Saved comparisons: {run}')
else:
    print('Inference disabled. Set CHECKPOINT and RUN_INFERENCE=True on the GPU machine.')
